# On-policy Prediction with Approximation

This chapter begins the study of function approximation in reinforcement learning by using it to estimate the state-value function from on-policy data: approximating $v_\pi$ from experience generated using a known policy $\pi$.

The approximate value function is not represented as a table. It is represented as a parameterized functional form with weight vector $\mathbf{w} \in \mathbb{R}^d$. The approximation is written

$$
\hat{v}(s, \mathbf{w}) \approx v_\pi(s),
$$

where $\hat{v}(s, \mathbf{w})$ is the approximate value of state $s$ given weight vector $\mathbf{w}$.

Examples of such parameterized forms include:

- A function linear in features of the state, where $\mathbf{w}$ is the vector of feature weights.
- A multi-layer artificial neural network, where $\mathbf{w}$ is the vector of connection weights in all layers.
- A decision tree, where $\mathbf{w}$ contains the numbers defining the split points and leaf values.

Typically, the number of weights is much less than the number of states:

$$
d \ll |\mathcal{S}|.
$$

Because changing one weight changes the estimated value of many states, updating a single state generalizes from that state to affect many other state values. This generalization can make learning more powerful, but also more difficult to manage and understand.


## 9.1 Value-function Approximation

Prediction methods update an estimated value function by shifting the estimate at particular states toward a backed-up value, or update target. An individual update is denoted

$$
s \mapsto u,
$$

where $s$ is the state updated and $u$ is the update target for the estimated value of that state. Examples include:

- Monte Carlo update: $S_t \mapsto G_t$.
- One-step TD update: $S_t \mapsto R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}_t)$.
- $n$-step TD update: $S_t \mapsto G_{t:t+n}$.
- DP expected update: $s \mapsto \mathbb{E}_\pi [R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w}_t) \mid S_t = s]$.

In the DP case, $s$ is an arbitrary state. In the other cases, the state is encountered in actual experience.

Each update can be interpreted as an example of the desired input-output behavior of the value function. The state $s$ is the input, and the target $u$ is the desired output. In the tabular case, the update has simply shifted the table entry for $s$ a fraction of the way toward $u$ and left all other state values unchanged.

With function approximation, the examples $s \mapsto u$ are passed to supervised learning methods for value prediction. Because the outputs are numbers, these methods are called function approximation methods. The function they approximate is interpreted as the estimated value function.

Conventional supervised learning assumes a static training set over which multiple passes are made. Reinforcement learning instead involves learning online, while the agent interacts with its environment or with a model of its environment. To be useful here, methods must learn efficiently from incrementally acquired data.

Reinforcement learning also requires methods for nonstationary target functions. In control methods based on GPI, generalized policy iteration seeks to learn $q_\pi$ while $\pi$ changes. Even if the policy remains the same, bootstrapping methods such as DP and TD can make target values nonstationary. Methods that cannot easily handle such nonstationarity are less suitable for reinforcement learning.


## 9.2 The Prediction Objective (VE)

With tabular methods, the learned value function can equal the true value function exactly. With function approximation, updating one state can affect many other states, so there may be no single setting of the weights that gives every state its exact correct value.

The objective must therefore specify which states matter most. A state distribution $\mu(s) \geq 0$, with $\sum_s \mu(s) = 1$, represents how much we care about the error in each state $s$. The error in state $s$ is the square of the difference between the approximate value and the true value. This gives the mean squared value error objective:

$$
\overline{\text{VE}}(\mathbf{w}) = \sum_{s \in \mathcal{S}} \mu(s) \left[v_\pi(s) - \hat{v}(s, \mathbf{w})\right]^2.
$$

The square root of this measure, the root VE, measures roughly how much the approximate values differ from the true values in often-used units. Often $\mu(s)$ is chosen as the fraction of time spent in state $s$. In continuing tasks, the on-policy distribution is the stationary distribution under $\pi$.

### On-policy Distribution in Episodic Tasks

In episodic tasks, the on-policy distribution depends on how initial states are chosen. Let $h(s)$ be the probability that an episode begins in state $s$, and let $\eta(s)$ be the expected number of time steps spent in state $s$ in a single episode. Time spent in state $s$ equals time from starting in $s$ plus time from transitions into $s$:

$$
\eta(s) = h(s) + \sum_{\bar{s}} \eta(\bar{s}) \sum_a \pi(a \mid \bar{s}) p(s \mid \bar{s}, a), \quad \text{for all } s \in \mathcal{S}.
$$

The on-policy distribution is the normalized expected time spent in each state:

$$
\mu(s) = \frac{\eta(s)}{\sum_{s'} \eta(s')}, \quad \text{for all } s \in \mathcal{S}.
$$

This is the natural choice without discounting. If there is discounting, $\gamma < 1$ should be treated as a form of termination by including a factor of $\gamma$ in the second term of the equation for $\eta(s)$.

The continuing and episodic cases behave similarly, but with approximation they must be treated separately in formal analyses.

### Optimality Under VE

VE is the performance objective used here, although it is not necessarily the best objective for the ultimate purpose of finding a better policy. For now, the focus is on VE.

An ideal goal is to find a global optimum, a weight vector $\mathbf{w}^*$ such that

$$
\overline{\text{VE}}(\mathbf{w}^*) \leq \overline{\text{VE}}(\mathbf{w}) \quad \text{for all possible } \mathbf{w}.
$$

For simple function approximators such as linear ones, this goal is sometimes possible. For complex function approximators such as artificial neural networks and decision trees, methods may seek only a local optimum, a weight vector $\mathbf{w}^*$ such that

$$
\overline{\text{VE}}(\mathbf{w}^*) \leq \overline{\text{VE}}(\mathbf{w})
$$

for all $\mathbf{w}$ in some neighborhood of $\mathbf{w}^*$. For nonlinear function approximation, there is often no guarantee of convergence to an optimum or even to within a bounded distance of an optimum. Some methods can drive VE toward infinity in the limit.

The framework is now: reinforcement learning generates training examples for function approximation, and VE is the performance measure the methods seek to minimize. The rest of the chapter focuses on function approximation methods based on gradient descent.


## 9.3 Stochastic-gradient and Semi-gradient Methods

Stochastic gradient descent methods are a widely used class of function approximation methods and are well suited to online reinforcement learning.

The weight vector is a column vector

$$
\mathbf{w} = (w_1, w_2, \ldots, w_d)^\top,
$$

and $\hat{v}(s, \mathbf{w})$ is differentiable in $\mathbf{w}$ for all $s \in \mathcal{S}$. At each discrete time step $t$, $\mathbf{w}_t$ is updated.

Assume first that, on each step, a new example $S_t \mapsto v_\pi(S_t)$ is observed, with $S_t$ chosen randomly according to $\mu$ and with the exact true value available. The gradient-descent update for minimizing VE is

$$
\mathbf{w}_{t+1} = \mathbf{w}_t - \frac{1}{2}\alpha \nabla \left[v_\pi(S_t) - \hat{v}(S_t, \mathbf{w}_t)\right]^2
$$

$$
= \mathbf{w}_t + \alpha \left[v_\pi(S_t) - \hat{v}(S_t, \mathbf{w}_t)\right]\nabla \hat{v}(S_t, \mathbf{w}_t),
$$

where $\alpha$ is a positive step-size parameter. The gradient of any scalar expression $f(\mathbf{w})$ is

$$
\nabla f(\mathbf{w}) = \left(\frac{\partial f(\mathbf{w})}{\partial w_1}, \frac{\partial f(\mathbf{w})}{\partial w_2}, \ldots, \frac{\partial f(\mathbf{w})}{\partial w_d}\right)^\top.
$$

This update is in the direction in which the error falls most rapidly. Because the overall step is proportional to the negative gradient of the example's squared error, it is called a gradient-descent method. Across many examples, making small steps makes the overall effect minimize average performance measures such as VE.

### Stochastic-gradient Update

The exact update cannot usually be performed because $v_\pi(S_t)$ is unknown. Instead, use an approximation $U_t$. This gives the stochastic-gradient method for state-value prediction:

$$
\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha \left[U_t - \hat{v}(S_t, \mathbf{w}_t)\right]\nabla \hat{v}(S_t, \mathbf{w}_t).
$$

If $U_t$ is an unbiased estimate, that is, if $\mathbb{E}[U_t \mid S_t = s] = v_\pi(s)$ for each $t$, then $\mathbf{w}_t$ is guaranteed to converge to a local optimum under standard conditions on decreasing $\alpha$.

For example, if the state examples are generated by interaction with the environment using policy $\pi$, then the true value of a state is the expected value of the return following it. The Monte Carlo target $U_t = G_t$ is an unbiased estimate of $v_\pi(S_t)$. With this choice, the general SGD method converges to a locally optimal approximation to $v_\pi$. This method is called gradient Monte Carlo state-value prediction.

```text
Gradient Monte Carlo Algorithm for Estimating v approximately equal to v_pi

Input: the policy to be evaluated
Input: a differentiable function vhat: S x R^d -> R
Algorithm parameter: step size alpha > 0
Initialize value-function weights w in R^d arbitrarily (e.g., w = 0)

Loop forever (for each episode):
    Generate an episode S_0, A_0, R_1, S_1, A_1, ..., R_T, S_T using pi
    Loop for each step of episode, t = 0, 1, ..., T - 1:
        w <- w + alpha [G_t - vhat(S_t, w)] grad vhat(S_t, w)
```

### Semi-gradient Methods

If a bootstrapping estimate of $v_\pi(S_t)$ is used as the target $U_t$, the same convergence guarantees are not obtained. Bootstrapping targets such as $n$-step returns $G_{t:t+n}$ or DP targets depend on the current value of the weight vector $\mathbf{w}_t$ through the estimate $\hat{v}$. The true gradient-descent update would take this dependency into account. The update above ignores the effect of changing $\mathbf{w}_t$ on the target, including only the effect on the estimate. For this reason, bootstrapping methods that use this update are called semi-gradient methods.

Although semi-gradient methods do not converge as robustly as gradient methods, they converge reliably in important cases. They also remain continual and online, without waiting for the end of an episode. The semi-gradient TD(0) method uses

$$
U_t = R_{t+1} + \gamma \hat{v}(S_{t+1}, \mathbf{w})
$$

as its target.

```text
Semi-gradient TD(0) for estimating v approximately equal to v_pi

Input: the policy pi to be evaluated
Input: a differentiable function vhat: S+ x R^d -> R such that vhat(terminal, .) = 0
Algorithm parameter: step size alpha > 0
Initialize value-function weights w in R^d arbitrarily (e.g., w = 0)

Loop for each episode:
    Initialize S
    Loop for each step of episode:
        Choose A ~ pi(.|S)
        Take action A, observe R, S'
        w <- w + alpha [R + gamma vhat(S', w) - vhat(S, w)] grad vhat(S, w)
        S <- S'
    until S is terminal
```

### State Aggregation

State aggregation is a simple form of generalizing function approximation in which states are grouped together with one estimated value for each group. The value of a state is estimated as its group's component, and the state is updated by changing only its group's component. In this special case of SGD, the gradient is $1$ for the updated state's group component and $0$ for the other components.

In the 1000-state random walk, states are numbered from 1 to 1000 and all episodes begin near the center, in state 500. Transitions are to one of the 100 neighboring states to the left or one of the 100 neighboring states to the right, all with equal probability. If the current state is near an edge, the transition may go beyond the edge and terminate on that side. Termination on the left gives reward $-1$, and termination on the right gives reward $+1$. All other transitions have reward $0$.

For the 1000-state task, state aggregation into 10 groups of 100 states each, using gradient Monte Carlo with a step size of $\alpha = 2 \times 10^{-5}$ for 100,000 episodes, produces an approximate value function typical of state aggregation: within each group the approximate value is constant, and it changes abruptly from one group to the next.

The approximate values are close to the global minimum of VE. The state distribution is shown with a right-side scale. State 500 is visited first in every episode, but is rarely visited again; about 1.37% of time steps are spent in the start state. States near the start are often not visited, whereas states near the extremes are encountered last before termination. The most visible effect of this distribution is on the leftmost group: its aggregate value is biased toward state 100, whose true value is higher than the true value of state 1.


## 9.4 Linear Methods

A central special case of function approximation is the linear case. Each state $s$ has a real-valued feature vector

$$
\mathbf{x}(s) = (x_1(s), x_2(s), \ldots, x_d(s))^\top,
$$

with the same number of components as $\mathbf{w}$. The approximate value is the inner product between $\mathbf{w}$ and $\mathbf{x}(s)$:

$$
\hat{v}(s, \mathbf{w}) = \mathbf{w}^\top \mathbf{x}(s) = \sum_{i=1}^d w_i x_i(s).
$$

The approximate value function is linear in the weights. The vector $\mathbf{x}(s)$ is called a feature vector representing state $s$, and each component $x_i(s)$ is a feature. The features form a linear basis for the set of approximate functions.

For linear methods, the gradient with respect to the weights is

$$
\nabla \hat{v}(s, \mathbf{w}) = \mathbf{x}(s).
$$

Thus the general SGD update becomes

$$
\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha \left[U_t - \hat{v}(S_t, \mathbf{w}_t)\right]\mathbf{x}(S_t).
$$

Linear SGD is favorable for mathematical analysis. In the linear case there is only one optimum, or in degenerate cases one set of equally good optima, and using an appropriate step-size schedule guarantees convergence to or near the global optimum. Gradient Monte Carlo with linear function approximation converges to the global optimum, or near it if $\alpha$ is reduced over time according to the usual conditions.

### Linear Semi-gradient TD(0)

Semi-gradient TD(0) also converges under linear function approximation, but this does not follow from general SGD convergence because the method is not true SGD. With $\mathbf{x}_t = \mathbf{x}(S_t)$, the update is

$$
\mathbf{w}_{t+1} = \mathbf{w}_t + \alpha \left(R_{t+1} + \gamma \mathbf{w}_t^\top \mathbf{x}_{t+1} - \mathbf{w}_t^\top \mathbf{x}_t\right)\mathbf{x}_t
$$

$$
= \mathbf{w}_t + \alpha \left(R_{t+1}\mathbf{x}_t - \mathbf{x}_t(\mathbf{x}_t - \gamma \mathbf{x}_{t+1})^\top \mathbf{w}_t\right).
$$

After the system reaches steady state, the expected next weight vector can be written

$$
\mathbb{E}[\mathbf{w}_{t+1} \mid \mathbf{w}_t] = \mathbf{w}_t + \alpha(\mathbf{b} - \mathbf{A}\mathbf{w}_t),
$$

where

$$
\mathbf{b} = \mathbb{E}[R_{t+1}\mathbf{x}_t] \in \mathbb{R}^d
$$

and

$$
\mathbf{A} = \mathbb{E}\left[\mathbf{x}_t(\mathbf{x}_t - \gamma \mathbf{x}_{t+1})^\top\right] \in \mathbb{R}^{d \times d}.
$$

If the system converges, it must converge to the weight vector $\mathbf{w}_{\text{TD}}$ satisfying

$$
\mathbf{b} - \mathbf{A}\mathbf{w}_{\text{TD}} = 0,
$$

so

$$
\mathbf{w}_{\text{TD}} = \mathbf{A}^{-1}\mathbf{b}.
$$

This quantity is the TD fixed point.

### Why Linear TD(0) Converges

The expected update can be rewritten as

$$
\mathbb{E}[\mathbf{w}_{t+1} \mid \mathbf{w}_t] = (\mathbf{I} - \alpha \mathbf{A})\mathbf{w}_t + \alpha \mathbf{b}.
$$

This is a diagonal matrix plus an off-diagonal matrix. If $\alpha$ is small enough, the diagonal terms remain less than one, and the off-diagonal terms do not prevent convergence. This depends on $\mathbf{A}$ being positive definite.

Let $\mathbf{D}$ be the $|\mathcal{S}| \times |\mathcal{S}|$ diagonal matrix with $\mu(s)$ on its diagonal, let $\mathbf{X}$ be the $|\mathcal{S}| \times d$ matrix whose rows are $\mathbf{x}(s)$, and let $\mathbf{P}$ be the state-transition matrix under policy $\pi$. Then

$$
\mathbf{A} = \mathbf{X}^\top \mathbf{D}(\mathbf{I} - \gamma \mathbf{P})\mathbf{X}.
$$

The proof assumes that all columns of $\mathbf{X}$ are linearly independent. It is enough to show that the key matrix $\mathbf{D}(\mathbf{I} - \gamma \mathbf{P})$ is positive definite, because then $\mathbf{A}$ is also positive definite. The argument uses the stationary distribution $\mu$:

$$
\mathbf{1}^\top \mathbf{D}(\mathbf{I} - \gamma \mathbf{P}) = \mu^\top(\mathbf{I} - \gamma \mathbf{P}) = \mu^\top - \gamma \mu^\top \mathbf{P} = (1 - \gamma)\mu^\top.
$$

All components of this vector are positive. The matrix also has positive diagonal entries, non-positive off-diagonal entries, positive row sums, and positive column sums. These facts imply that the key matrix and $\mathbf{A}$ are positive definite. Under the on-policy TD(0) update distribution and an appropriate schedule for reducing $\alpha$ over time, convergence with probability one follows.

At the TD fixed point, in the continuing case, the VE is within a bounded expansion of the lowest possible error:

$$
\overline{\text{VE}}(\mathbf{w}_{\text{TD}}) \leq \frac{1}{1 - \gamma} \min_{\mathbf{w}} \overline{\text{VE}}(\mathbf{w}).
$$

This bound says the asymptotic error of TD can be no more than $\frac{1}{1-\gamma}$ times the smallest possible error. Because $\gamma$ is often near one, this expansion factor can be large, so the bound is a worst-case guarantee. In practice, TD methods are often much better than this bound suggests.

### Bootstrapping with State Aggregation

State aggregation is a special case of linear function approximation, so the linear TD convergence results apply. In the 1000-state random walk, a state-aggregation version of TD produces values shifted farther from the true values than the Monte Carlo approximation, but it learns faster.

To obtain similar visual results with function approximation, the aggregation was changed to 20 groups of 50 states each. The performance of $n$-step methods is qualitatively similar to the tabular case, but the RMS error is measured as an unweighted average over all states and over the first 10 episodes instead of as a VE objective.

The semi-gradient $n$-step TD algorithm is the natural extension of tabular $n$-step TD to function approximation. It stores states and rewards in arrays indexed modulo $n+1$.

```text
n-step semi-gradient TD for estimating v approximately equal to v_pi

Input: the policy pi to be evaluated
Input: a differentiable function vhat: S+ x R^d -> R such that vhat(terminal, .) = 0
Algorithm parameters: step size alpha > 0, a positive integer n
Initialize value-function weights w arbitrarily (e.g., w = 0)
All store and access operations on S_i and R_i can take their index mod n + 1

Loop for each episode:
    Initialize and store S_0 != terminal
    T <- infinity
    Loop for t = 0, 1, 2, ...:
        If t < T, then:
            Take an action according to pi(.|S_t)
            Observe and store the next reward as R_{t+1} and the next state as S_{t+1}
            If S_{t+1} is terminal, then T <- t + 1
        tau <- t - n + 1
        If tau >= 0:
            G <- sum_{i=tau+1}^{min(tau+n,T)} gamma^{i-tau-1} R_i
            If tau + n < T, then G <- G + gamma^n vhat(S_{tau+n}, w)
            w <- w + alpha [G - vhat(S_tau, w)] grad vhat(S_tau, w)
    Until tau = T - 1
```

The key equation of this algorithm is

$$
\mathbf{w}_{t+n} = \mathbf{w}_{t+n-1} + \alpha\left[G_{t:t+n} - \hat{v}(S_t, \mathbf{w}_{t+n-1})\right]\nabla \hat{v}(S_t, \mathbf{w}_{t+n-1}), \quad 0 \leq t < T,
$$

where the $n$-step return is generalized to

$$
G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1}R_{t+n} + \gamma^n \hat{v}(S_{t+n}, \mathbf{w}_{t+n-1}), \quad 0 \leq t \leq T - n.
$$


## 9.5 Feature Construction for Linear Methods

Linear methods are useful not only because their convergence properties are well understood, but also because they can be efficient in data and computation. Their quality depends critically on how states are represented as feature vectors.

The feature vector can be constructed from the state-space geometry, intuition, or a combination of both. In linear methods, features can add prior knowledge because their values determine which states generalize to which other states.

A limitation of the linear form is that it cannot account for interactions between features unless those interactions are included as features. For example, if two features are height and angular velocity, then the function may also need their product as a separate feature.

### 9.5.1 Polynomials

For states initially represented as numbers, polynomial-basis features are one simple construction. They take powers and products of the original state dimensions so that the approximation remains linear in the weights while representing higher-order interactions among state dimensions.

Suppose each state $s$ corresponds to $k$ numbers, $s_1, s_2, \ldots, s_k$, with each $s_i \in \mathbb{R}$. For each $k$-dimensional tuple $c_i$, each order-$n$ polynomial-basis feature $x_i$ can be written as

$$
x_i(s) = \prod_{j=1}^k s_j^{c_{i,j}},
$$

where each $c_{i,j}$ is an integer in $\{0, 1, \ldots, n\}$. These features make up the order-$n$ polynomial basis for dimension $k$, which contains $(n+1)^k$ different features.

Higher-order polynomial bases can represent more complicated functions, but the number of features grows exponentially with the dimensionality of the state space. This limits their practical use in high-dimensional reinforcement learning tasks unless the degree is low or the dimensionality is reduced by prior knowledge or automated feature selection.

### 9.5.2 Fourier Basis

Fourier basis functions use sinusoidal features of different frequencies. For continuous state variables normalized to $[0, 1]$, the order-$n$ Fourier cosine basis is

$$
x_i(s) = \cos(\pi s^\top c^i),
$$

where $c^i = (c^i_1, \ldots, c^i_k)^\top$, with $c^i_j \in \{0, \ldots, n\}$ for $j=1, \ldots, k$ and $i=1, \ldots, (n+1)^k$.

Fourier cosine features can perform well compared with polynomial and radial basis functions. They can also have trouble with discontinuities because the fit of discontinuous functions requires many high-frequency basis functions.

The number of Fourier basis features grows exponentially with the dimension of the state space. For high-dimensional state spaces, it is necessary to select a subset of the features. This can be done with prior knowledge or automatically, but in general the book does not recommend using polynomial or Fourier features for online learning.

### 9.5.3 Coarse Coding

Coarse coding represents a state by a set of overlapping regions, often shown as circles in continuous state space. Each feature corresponds to one region and has value $1$ if the state is inside the region and $0$ otherwise. Such a binary feature is either present or absent.

With linear gradient-descent function approximation, only weights for active features are updated. Therefore the size and shape of the receptive fields determine how generalization occurs. Large receptive fields produce broad generalization; small receptive fields produce narrow generalization; asymmetric receptive fields produce asymmetric generalization.

The number of active features can influence early learning. With broader features, initial generalization can be strong but may hurt final accuracy. With narrower features, learning may generalize less initially. Receptive-field shape tends to have a strong effect on generalization but little effect on asymptotic solution quality.

### 9.5.4 Tile Coding

Tile coding is a form of coarse coding for multidimensional continuous spaces that is flexible and computationally efficient. Each partition of the state space is a tiling, and each element of a partition is a tile. Each state is represented by the set of tiles that contain it.

A single grid tiling produces simple state aggregation. To obtain overlapping receptive fields, tile coding uses multiple offset tilings. If there are $n$ tilings, then exactly $n$ binary features are active at any time. This makes computation efficient because only active weights are involved in the update.

Tile coding can improve early learning compared with one-tiling state aggregation. Choosing a step size such as $\alpha = 1/n$ has an intuitive effect: if all active weights are initially zero and the target is one, then the next estimate moves directly to the target. Smaller steps move more slowly and maintain prior estimates more strongly.

Generalization occurs to states that share active tiles. Multiple offset tilings avoid the blocky artifacts of a single tiling and provide a richer set of possible generalizations. If the offsets are asymmetric, then the generalization patterns also become asymmetric.

Hashing can be used to reduce memory requirements. The active tiles are mapped through a hashing function to a lower-dimensional memory space. This can greatly reduce memory use, but distinct tiles may share the same component, so hash collisions may cause unintended generalization.

### 9.5.5 Radial Basis Functions

Radial basis functions are the natural generalization of coarse coding to continuous-valued features. Instead of binary features, RBFs produce graded responses that vary smoothly and are differentiable.

A typical RBF feature $x_i$ depends on the distance between the state $s$ and the feature's prototypical or center state $c_i$, relative to the feature's width $\sigma_i$:

$$
x_i(s) = \exp\left(-\frac{\lVert s - c_i \rVert^2}{2\sigma_i^2}\right).
$$

The norm can be chosen in whatever way seems most appropriate to the state and task. RBFs can approximate functions that vary smoothly and are differentiable, but can require substantial additional computation. In high-dimensional state spaces, there are often many more important states than can be represented by a practical number of RBF centers.


## 9.6 Selecting Step-Size Parameters Manually

Most SGD methods require a designer to choose a step-size parameter $\alpha$. Although stochastic approximation theory gives conditions on slowly decreasing step-size sequences that guarantee convergence, these conditions tend to produce learning that is too slow.

The classical choice $\alpha = 1/t$, which gives sample averages in tabular Monte Carlo methods, is not appropriate for TD methods, nonstationary problems, or methods using function approximation. Optimal matrix step-size methods can be extended to temporal-difference learning, but they require $O(d^2)$ step-size parameters, or $d$ times more parameters than are learned, so they are ruled out for large problems.

The tabular case gives intuition. With $\alpha = 1$, the sample error is completely eliminated after one target. Usually learning should be slower. With $\alpha = 1/10$, a tabular estimate takes about 10 experiences to converge approximately to the mean target; with $\alpha = 1/100$, it takes about 100 experiences. In general, if $\alpha = 1/\tau$, then the estimate approaches the mean of its targets, with the $\tau$ most recent targets having the greatest effect.

With general function approximation there is no clear number of experiences with a state, because each state can be similar or dissimilar to many other states. For linear function approximation, an analogous rule is to choose $\alpha$ based on how many experiences should be needed to learn about a feature vector. If a good scale is $\tau$ experiences, then a useful rule of thumb for linear SGD is

$$
\alpha \doteq \left(\tau \mathbb{E}[\mathbf{x}^\top \mathbf{x}]\right)^{-1},
$$

where $\mathbf{x}$ is a random feature vector drawn from the same distribution as the input vectors used in SGD. This works best when feature-vector lengths do not vary greatly, ideally when $\mathbf{x}^\top \mathbf{x}$ is constant.


## 9.8 Least-Squares TD

The methods discussed so far require computation per time step proportional to the number of parameters. With more computation, one can do better for linear function approximation. Least-Squares TD, or LSTD, computes estimates of the TD fixed-point quantities and then computes the fixed point directly.

For linear TD(0), the TD fixed point is

$$
\mathbf{w}_{\text{TD}} = \mathbf{A}^{-1}\mathbf{b},
$$

where

$$
\mathbf{A} \doteq \mathbb{E}\left[\mathbf{x}_t(\mathbf{x}_t - \gamma \mathbf{x}_{t+1})^\top\right]
$$

and

$$
\mathbf{b} \doteq \mathbb{E}[R_{t+1}\mathbf{x}_t].
$$

LSTD forms the natural estimates

$$
\hat{\mathbf{A}}_t \doteq \sum_{k=0}^{t-1} \mathbf{x}_k(\mathbf{x}_k - \gamma \mathbf{x}_{k+1})^\top + \varepsilon\mathbf{I}
$$

and

$$
\hat{\mathbf{b}}_t \doteq \sum_{k=0}^{t-1} R_{k+1}\mathbf{x}_k,
$$

where $\mathbf{I}$ is the identity matrix and $\varepsilon\mathbf{I}$, for small $\varepsilon > 0$, ensures that $\hat{\mathbf{A}}_t$ is always invertible. The LSTD estimate of the TD fixed point is

$$
\mathbf{w}_t \doteq \hat{\mathbf{A}}_t^{-1}\hat{\mathbf{b}}_t.
$$

This algorithm is data efficient, but computationally expensive. A naive implementation requires inverting a $d \times d$ matrix at each time step, with general inverse computation in $O(d^3)$.

### Incremental Inverse Update

The inverse of $\hat{\mathbf{A}}_t$ can be updated incrementally with only $O(d^2)$ computations using the Sherman-Morrison formula:

$$
\hat{\mathbf{A}}_t^{-1} = \hat{\mathbf{A}}_{t-1}^{-1} - \frac{\hat{\mathbf{A}}_{t-1}^{-1}\mathbf{x}_{t-1}(\mathbf{x}_{t-1} - \gamma\mathbf{x}_t)^\top \hat{\mathbf{A}}_{t-1}^{-1}}{1 + (\mathbf{x}_{t-1} - \gamma\mathbf{x}_t)^\top \hat{\mathbf{A}}_{t-1}^{-1}\mathbf{x}_{t-1}},
$$

for $t > 0$, with $\hat{\mathbf{A}}_0 \doteq \varepsilon\mathbf{I}$. The algorithm then updates $\hat{\mathbf{A}}^{-1}$, $\hat{\mathbf{b}}$, and $\mathbf{w}$ online.

```text
LSTD for estimating vhat = w^T x approximately equal to v_pi (O(d^2) version)

Input: feature representation x: S+ -> R^d such that x(terminal) = 0
Algorithm parameter: small epsilon > 0

Ainv <- epsilon^-1 I          A d x d matrix
bhat <- 0                     A d-dimensional vector

Loop for each episode:
    Initialize S; x <- x(S)
    Loop for each step of episode:
        Choose and take action A ~ pi(.|S), observe R, S'; x' <- x(S')
        v <- Ainv (x - gamma x')
        Ainv <- Ainv - (Ainv x) v^T / (1 + v^T x)
        bhat <- bhat + R x
        w <- Ainv bhat
        S <- S'; x <- x'
    until S' is terminal
```

LSTD is significantly more expensive than $O(d)$ semi-gradient TD, but it is often data efficient. It does not require a step-size parameter, but because it effectively uses the inverse of all previous data, it has no forgetting mechanism. In control applications, this can be inconvenient because changing the target policy changes the optimal approximation.


## 9.9 Memory-based Function Approximation

Parametric methods summarize all observations in the parameters of a function form. After an observation has been used to update the parameters, the training example can be discarded.

Memory-based function approximation methods instead save training examples in memory. When queried at a state, they retrieve relevant examples and use them to compute a value estimate. Because most processing is postponed until a query occurs, these methods are often called lazy learning methods.

Memory-based methods are nonparametric: the approximate function is not limited to a fixed parameterized class. It is determined by the stored training examples. If memory or storage is available, the system can use more examples and produce increasingly accurate approximations.

### Local Approximation

Many memory-based methods are local methods: they approximate the value function only near the current query state. They reserve a set of training examples judged relevant to the query state, usually by a distance between states. The closer a training example's state is to the query state, the more relevant it is. After a value is given for the query state, the local approximation is discarded.

The nearest neighbor method finds the example in memory whose state is closest to the query state and returns that example's value as the approximate value of the query state. Weighted average methods retrieve a set of nearest examples and return a weighted average of their target values, with weights generally decreasing as distance increases. Locally weighted regression fits a surface to nearby states by minimizing a weighted error measure, where weights depend on distance from the query state; the returned value is the surface evaluated at the query state.

### Advantages and Costs

Compared with parametric methods, memory-based methods do not require limiting approximations to pre-specified functional forms. Memory-based local approximation methods are well suited for reinforcement learning because target functions change, and because local neighborhoods of state or state-action pairs visited in real or simulated trajectories can be used without requiring a good global approximation.

Memory-based methods also address the curse of dimensionality differently. A tabular approximation over a $k$-dimensional state with $n$ possible values per dimension requires memory exponential in $k$. For a memory-based method, each example requires memory proportional to $k$, so memory is linear in $k$ for each stored example.

The main practical issue is query speed. A naive search for the $k$ nearest neighbors in a large database can take too long. Acceleration methods include parallel computation, special-purpose hardware, and multidimensional data structures such as k-d trees, which recursively split the state space and limit the search to relevant regions. These methods are most feasible when neighbors lie in low-dimensional areas of the state space.

Locally weighted regression also requires fast local regression computations, which must be repeated to answer each query. Researchers have developed ways to address these problems, including methods for forgetting entries to keep database size within bounds.


## 9.10 Kernel-based Function Approximation

Memory-based methods such as weighted average and locally weighted regression depend on assigning weights to examples according to their distance or similarity to a query state. The function that assigns these weights is called a kernel function, or simply a kernel.

In weighted average and locally weighted regression methods, a kernel function

$$
k : \mathbb{R} \to \mathbb{R}
$$

assigns weights to distances between states. More generally, weights do not have to depend only on distance; they can depend on another measure of similarity. In this case,

$$
k : \mathcal{S} \times \mathcal{S} \to \mathbb{R},
$$

so $k(s, s')$ is the weight given to data about $s'$ in answering queries about $s$. The value $k(s, s')$ measures the strength of generalization from $s'$ to $s$.

Kernel functions numerically express how relevant knowledge about one state is to another state. Different choices of kernel produce different patterns of generalization.

### Kernel Regression

Kernel regression is a memory-based method that computes a kernel-weighted average of the targets of all examples stored in memory. If $\mathcal{D}$ is the set of stored examples and $g(s')$ denotes the target for state $s'$, then kernel regression approximates the target function, here a value function depending on $\mathcal{D}$, as

$$
\hat{v}(s, \mathcal{D}) = \sum_{s' \in \mathcal{D}} k(s, s')g(s').
$$

The weighted average method is a special case in which $k(s, s')$ is non-zero only when $s$ and $s'$ are close enough that the sum need not be computed over all of $\mathcal{D}$.

A common kernel is the Gaussian radial basis function used in RBF function approximation. In that case, RBFs are fixed centers and widths, each center has an associated weight, and learning adjusts the weights. Kernel regression with an RBF kernel differs in two ways: first, it is memory-based because the RBFs are centered on the states of the stored examples; second, it is nonparametric because there are no parameters other than the stored data.

### Kernel Trick

A linear parametric regression method whose state $s$ is represented by a feature vector

$$
\mathbf{x}(s) = (x_1(s), x_2(s), \ldots, x_d(s))^\top
$$

can be recast as kernel regression with kernel

$$
k(s, s') = \mathbf{x}(s)^\top \mathbf{x}(s').
$$

Kernel regression with this kernel produces the same approximation that a linear parametric method would if it used these feature vectors and learned with the training data.

The kernel can sometimes be computed without explicitly constructing the feature vectors. This can provide advantages over the equivalent parametric method. In some cases, the feature vector is very high dimensional, but the kernel function can be computed efficiently and directly from the states. This is the kernel trick.


## 9.11 Looking Deeper at On-policy Learning: Interest and Emphasis

The algorithms considered so far have treated all encountered states equally, as if they were all equally important. Sometimes some states are more interesting than others, and limited function approximation resources should then be used more where they matter most.

Previously, updating according to the on-policy distribution generalized results over available resources: if some states occur more often under the target policy, the approximation becomes better for those states. This is not the same as learning according to an arbitrary distribution of interest.

An interest function is introduced as a random variable $I_t$ called interest, which indicates the degree to which we are interested in accurately valuing the state-action pair at time $t$. Usually $I_t$ is set from the state at time $t$; if one does not care about a state, its interest should be zero, and if one fully cares, it might be one.

The interest can be used to define a distribution $\mu$, for example as the fraction of time a state is encountered multiplied by the interest at that time. This distribution is used in the VE objective. If the learning update were simply multiplied by the interest, the standard update would become

$$
\mathbf{w}_{t+n} = \mathbf{w}_{t+n-1} + \alpha I_t \left[G_{t:t+n} - \hat{v}(S_t, \mathbf{w}_{t+n-1})\right]\nabla \hat{v}(S_t, \mathbf{w}_{t+n-1}), \quad 0 \leq t < T.
$$

Emphatic temporal-difference methods replace this interest term with an emphasis $M_t$. The update becomes

$$
\mathbf{w}_{t+n} = \mathbf{w}_{t+n-1} + \alpha M_t \left[G_{t:t+n} - \hat{v}(S_t, \mathbf{w}_{t+n-1})\right]\nabla \hat{v}(S_t, \mathbf{w}_{t+n-1}), \quad 0 \leq t < T,
$$

with the $n$-step return as in (9.16). The emphasis is determined recursively from the interest by

$$
M_t = I_t + \gamma^n M_{t-n}, \quad 0 \leq t < T,
$$

with $M_t \doteq 0$ for all $t < 0$. For Monte Carlo, where $G_{t:t+n} = G_t$, all updates are made at the end of the episode, $n = T - t$, and $M_t = I_t$.

### Example: Interest and Emphasis

In the four-state Markov reward process, episodes start in the leftmost state and transition once to the right at each step until the terminal state is reached. Rewards are $+1$ on each transition. The true values are $4, 3, 2, 1$, but the parameterization has only two components: one for the first two states and one for the last two states.

With ordinary gradient Monte Carlo or semi-gradient TD methods, the approximate values converge to the average of the true values in each pair, producing $3.5$ for the first two states and $1.5$ for the last two states.

If only the leftmost state has interest $1$ and all other states have interest $0$, then gradient Monte Carlo with interest converges to the parameter vector $\mathbf{w}=(4,0)$, giving the first state the correct value. Semi-gradient TD methods with interest and emphasis also converge to $\mathbf{w}=(4,2)$, giving correct values for the first and third states. The first state bootstraps from the third state, even though no update is made for the second or fourth states.
